In [ ]:
# Imports
from model_handler.resnet import resnet
from dataset_handler.pascal import PascalDataset
from heatmap_handler import get_lrp_methods
from model_handler import device
from utils.utils import SquareCropAndResize, imagenet_transform, inverse_transform, sam_predict
from prompt_handler.max_value import MaxValue
from prompt_handler.dummy import Dummy
from segment_anything import sam_model_registry, SamPredictor
from PIL import Image
from utils.xai_visualizer import Visualizer
from segment_eval_handler.iou import IOU
from heatmap_handler.lrp import simpleLRP
from heatmap_handler.lrp import simpleCRP


In [ ]:
# Handler Loader
model = resnet(4)
dataset = PascalDataset(device, imagenet_transform, inverse_transform, target_transform=None)
xai_methods = [simpleCRP]
point_generator = MaxValue()
segment_eval = IOU()

In [ ]:
# SAM
sam_checkpoint = 'sam_models\sam_vit_b_01ec64.pth'
model_type = "vit_b"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam = sam.to(device)

In [ ]:
SAMPLES = 3
MULTIPLE_RESULTS = True

data = dataset.get_samples(SAMPLES)
for i in data:
    predictor = SamPredictor(sam)
    predictor.set_image(i.original_image)
    i.sam_predictor = predictor

In [ ]:

for indx, entry in enumerate(data):
    # init xai method
    method = xai_methods[0](model)
    # generate initial heatmap (and model classification label)
    method(entry)
    # generate points prompts (updates heatmap if interpolated)
    point_generator(entry)
    sam_predict(entry, MULTIPLE_RESULTS)
    # print(entry)
    segment_eval(entry)
    print(entry)
    # print(entry)




In [ ]:
v = Visualizer(data)